# Stage 3: the development year

**Reads:** `data/gravity_clean.csv`, saved by stage 2.

**Writes:**
- `results/dev_runs.csv`: one row per configuration and training draw, with its test deviance, the other scores, its training deviance, its seconds, what the `ppml` fit of the draw dropped or left out, for `ppml` its coefficients, and for `ridge` the penalty its cross-validation chose (`ridge_alpha`).
- `results/dev_components.csv`: one row per training size and configuration, with the mean test deviance split into variance and remainder.

Both are read by stage 4. In quick mode, `run_all.py` also runs `03b_checkpoint.ipynb` on them, to test its code.

The stage runs every training size at once (section 6 of `docs/build_plan.md`). At each size, every draw takes one training sample from the training pool, fits every model on it (`ppml`, six forests, one boosted model for each depth, `mean`, `tree`, and the ridge line `ridge`, added by Yu at the checkpoint, decision 20) and scores every configuration on the same test set (section 1 of the plan gives the counts, and decision 19 the grid). Every draw is keyed by year, split, size and draw, so it gives the same numbers whichever worker runs it.

The table keeps the zero trade flows (decision 24). With zeros, an indicator can separate a training sample: every row on one side of it has a zero flow, and the `ppml` estimate does not exist. The fits of `ppml` and `ridge` leave those rows out (decision 25); the trees use every row. The first run of the project, on the table without zeros, ran this stage in two parts around the checkpoint: part 1 (`python code/run_all.py --part 1`, the sizes up to `CHECKPOINT_MAX`), then every size, which checked that it repeated the rows of part 1. The run with the zero flows has no part 1, so it compares nothing with part 1.

*Drafted with Claude (Anthropic), which was asked for stage 3 of the build plan: the test split, the training draws at every size with the 13 fits of each, the rows of `dev_runs.csv`, the split of the error into variance and remainder in `dev_components.csv`, the checks of section 8 that belong to this stage, and the comparison of the full run with part 1. It replaces the placeholder of 18 October 2026. Later asked for (21 October 2026, decision 19, "Apply"; prepared on 20 October before Yu's answer): the text made independent of the size of the grid that the checkpoint extended. Later asked for (21 October 2026, decision 20, "Add it"): the ridge line fitted on every draw, and the penalty it chose printed for each size. Later asked for (23 October 2026, stage 6): the loop over the sizes and draws moved into `run_experiment` in `helpers.py`, which stage 6 runs too, with the same output. Later asked for (4 November 2026, decisions 24 and 25, "Rerun with zeros" and "Drop the pairs"): the run with the zero flows, with no part 1 and no comparison with it, the countries of the test set counted in the training pool, the check of the rule of decision 25 on a toy sample, and the check of the deviance at a zero flow on the first draw.*

In [ ]:
from helpers import (CHECKPOINT_MAX, CONFIGURATIONS, DATA, DEV_YEAR, DRAWS, GBM_DEPTHS, GBM_TREE_COUNTS, N_JOBS, PART,
                     PREDICTORS, QUICK, RESULTS, SIZES, TARGET, check_separation_rule, countries, draw_sample, make_gbm,
                     projected_seconds, run_draw, run_experiment, seed_int, split_pairs, test_groups, unit_deviance)

import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_poisson_deviance

# Part 1 runs the sizes up to CHECKPOINT_MAX; without ADA_PART every size runs.
sizes = [n for n in SIZES if PART is None or n <= CHECKPOINT_MAX]
if PART is not None:
    print(f"Part {PART}, the training sizes up to {CHECKPOINT_MAX}:", sizes)
elif QUICK:
    print("Quick mode, every quick-mode training size:", sizes)
else:
    print("Full run, every training size:", sizes)
print(f"{DRAWS} draws per size, {len(CONFIGURATIONS)} configurations per draw, {N_JOBS} worker(s)")

# The base experiment of the development year uses split 0; stage 6 repeats it with split 1.
VARIANT = "base"
SPLIT = 0
RUNS = RESULTS / "dev_runs.csv"
COMPONENTS = RESULTS / "dev_components.csv"

## Step 1. Split the development year into a training pool and a test set

A random quarter of the unordered country pairs goes to the test set, with all its rows (`split_pairs` in `helpers.py`). Both directions of a pair, such as France to Italy and Italy to France, land on the same side, so no model is tested on the mirror image of a row it was trained on. The training samples of every size are drawn from the pool; the test set stays the same for every size, draw and model.

The split is by pair, not by country, so a country of a test pair can have other pairs in the pool, and a model can see that country in training. The TA asked that the report note this. The notebook prints how many countries of the test set are also in the pool; a training sample of n rows holds at most 2n countries.

In [ ]:
# Country identifiers are text, as in stage 2: no value is read as missing.
table = pd.read_csv(DATA / "gravity_clean.csv", dtype={"country_id_o": str, "country_id_d": str, "pair_id": str},
                    keep_default_na=False, na_values=[""])
pool, test = split_pairs(table, DEV_YEAR, SPLIT)
print(f"{DEV_YEAR}: {(table['year'] == DEV_YEAR).sum():,} rows")
print(f"  training pool: {len(pool):,} rows, {pool['pair_id'].nunique():,} pairs")
print(f"  test set:      {len(test):,} rows, {test['pair_id'].nunique():,} pairs")
test_countries = countries(test)
print(f"  countries in the test set: {len(test_countries)}, of which {len(test_countries & countries(pool))} "
      "are also in the training pool")

X_pool = pool[PREDICTORS].to_numpy(dtype=float)
y_pool = pool[TARGET].to_numpy(dtype=float)
# The ridge line keeps both directions of a pair in one fold of its cross-validation.
pair_pool = pool["pair_id"].to_numpy()
X_test = test[PREDICTORS].to_numpy(dtype=float)
y_test = test[TARGET].to_numpy(dtype=float)

## Step 2. Check the largest training size

The largest size has to be at most a quarter of the pool (section 6, stage 3 step 2). If it is not, the stage stops here: the size is not shrunk silently.

In [ ]:
largest = max(sizes)
assert largest <= len(pool) / 4, f"The largest size, {largest:,}, is over a quarter of the pool ({len(pool):,} rows). Stop and report."
print(f"Largest size {largest:,}: {largest / len(pool):.1%} of the pool")

## Three groups of test rows

A few very large flows can carry most of the deviance. To see where each model's error comes from, the test rows are cut once into three groups by the size of their flow: the largest 1%, the next 9% and the rest. Each draw records the deviance of each group, summed over its rows and divided by the number of test rows, so the three add up to the test deviance (stage 3 step 5).

In [ ]:
groups = test_groups(y_test)
for group, name in enumerate(["largest 1%", "next 9%", "rest"]):
    in_group = groups == group
    print(f"  {name:<10}: {in_group.sum():>5,} test rows, {y_test[in_group].sum() / y_test.sum():.1%} of the test flow")

## Check: staged prediction of the boosted trees (section 8)

Each boosted model is fitted once with the largest number of trees, and its predictions after fewer trees are read off the same fit (`staged_predict`). On the first draw, the prediction after the smallest number of trees is compared with a model fitted with only that many trees. They must be equal.

In [ ]:
n, draw = sizes[0], 0
positions = draw_sample(len(y_pool), DEV_YEAR, SPLIT, n, draw)
seed = seed_int("model", DEV_YEAR, SPLIT, n, draw)
fewest = GBM_TREE_COUNTS[0]
for depth in GBM_DEPTHS:
    staged = make_gbm(depth, seed).fit(X_pool[positions], y_pool[positions])
    for trees, staged_prediction in enumerate(staged.staged_predict(X_test), start=1):
        if trees == fewest:
            break
    fresh = make_gbm(depth, seed, trees=fewest).fit(X_pool[positions], y_pool[positions])
    assert np.array_equal(staged_prediction, fresh.predict(X_test)), f"Staged prediction differs from a fresh fit at depth {depth}."
print(f"n = {n}, draw {draw}: the staged prediction after {fewest} trees equals a fresh fit with {fewest} trees at every depth")

## Check: the rule of decision 25 on a toy sample (section 8)

`separated_rows` in `helpers.py` finds the indicators that separate a training sample and the rows on their separated side, which the fits of `ppml` and `ridge` leave out. On a toy sample of six rows, two of them with a zero flow, it must leave out exactly the rows of an indicator whose 1s all have a zero flow, none when one of those flows is positive, and the rows of an indicator whose 0s all have a zero flow.

In [ ]:
check_separation_rule()

## Check: the deviance at a zero flow (section 8, decision 24)

At a zero flow, the unit deviance `d(0, m) = 2m`: the term `y ln(y / m)` is taken as 0. `unit_deviance` in `helpers.py` gives the deviance of the three groups of test rows and the remainder of the split of the error (step 7). On the first draw, for every configuration, its mean over the test rows, zeros included, must equal scikit-learn's `mean_poisson_deviance` to a relative 1e-12.

In [ ]:
n, draw = sizes[0], 0
first = run_draw(X_pool, y_pool, pair_pool, X_test, y_test, DEV_YEAR, SPLIT, n, draw)
largest_difference = 0.0
for configuration, predicted in zip(CONFIGURATIONS, first["test_predicted"]):
    from_helpers = unit_deviance(y_test, predicted).mean()
    from_sklearn = mean_poisson_deviance(y_test, predicted)
    difference = abs(from_helpers - from_sklearn) / from_sklearn
    assert difference <= 1e-12, (f"{configuration['config']}: unit_deviance gives {from_helpers}, "
                                 f"mean_poisson_deviance {from_sklearn}. Stop and report.")
    largest_difference = max(largest_difference, difference)
print(f"n = {n}, draw {draw}: {int((y_test == 0).sum()):,} of the {len(y_test):,} test flows are zero; for all "
      f"{len(CONFIGURATIONS)} configurations the mean of unit_deviance equals mean_poisson_deviance "
      f"(largest relative difference {largest_difference:.1e})")

## Step 3. Time one draw

One draw at the largest size of this run is timed, its fits and predictions together (`run_draw`), and the minutes of this run of the stage (part 1 only, in part 1) are projected from it with the ratios between sizes measured in section 9 of the plan. The notebook does not wait: before the full run in Nuvolos, the projection of the checkpoint (item 8) is read, and Yu is asked if it is over 30 minutes.

In [ ]:
start = time.perf_counter()
run_draw(X_pool, y_pool, pair_pool, X_test, y_test, DEV_YEAR, SPLIT, largest, 0)
timing = time.perf_counter() - start
projected = sum(DRAWS * projected_seconds(timing, largest, n) for n in sizes) / 60
print(f"One draw at n = {largest:,}: {timing:.1f} seconds")
print(f"Projected time of this run of stage 3: {projected:.0f} minutes on one worker"
      + (f", about {projected / N_JOBS:.0f} with {N_JOBS} workers" if N_JOBS > 1 else ""))

## Steps 4 to 7. The draws

The loop over the sizes and the draws is `run_experiment` in `helpers.py`, which stage 6 runs too. For every size and every draw (`run_draw`):
- n rows are sampled from the pool without replacement;
- every model is fitted on that one sample, and every configuration predicts the test rows. If an indicator separates the sample, `ppml` leaves out the rows on its separated side and drops the indicator (decision 25). The ridge line (`FIT_RIDGE`, decision 20) first picks its penalty by 5-fold cross-validation inside the sample, with the folds keeping both directions of a pair together, then is refitted on the whole sample at that penalty (`fit_ridge`, section 6 of the plan, Ridge line); each of its fits leaves out the rows an indicator separates in the rows of that fit, as `ppml` does;
- every prediction is checked to be positive and finite;
- one row per configuration is kept for `dev_runs.csv` (`draw_rows`). Its seconds are those of the fit alone, as stage 3 step 5 defines them; for `gbm`, those of the one fit at its depth, and for `rf` they include the out-of-bag predictions that scikit-learn computes inside the fit; for `ridge`, those of its 31 fits summed.

The predictions themselves are not saved. For every configuration and test row, `run_experiment` keeps the running sum of the predictions over the draws and the running sum of their logs. After the last draw of a size, these give the split of the mean test deviance into variance and remainder (`split_error`, section 7), one row per configuration for `dev_components.csv`.

After each size, the notebook prints the checks on `ppml` of section 8: how many fits scikit-learn warned had fallen back from Newton-Cholesky to L-BFGS, how many draws had an indicator that was constant in the sample (and so dropped from that fit), how many draws had an indicator that separated the sample, with the rows left out of the fit for that reason in all (decision 25), and the largest relative difference between the predicted and the observed training total, over the rows the fit used, which unpenalised PPML keeps at the level of rounding errors unless a fit fell back (section 8). It also prints, for each penalty of the grid, in how many draws the cross-validation of the ridge line chose it.

In [ ]:
runs, components = run_experiment(X_pool, y_pool, pair_pool, X_test, y_test, VARIANT, DEV_YEAR, SPLIT, sizes, DRAWS)
print(f"dev_runs.csv: {len(runs):,} rows; dev_components.csv: {len(components):,} rows")

## Write the two files

In [ ]:
RESULTS.mkdir(exist_ok=True)
runs.to_csv(RUNS, index=False, lineterminator="\n")
components.to_csv(COMPONENTS, index=False, lineterminator="\n")
print(f"Wrote {RUNS.relative_to(RESULTS.parent)} and {COMPONENTS.relative_to(RESULTS.parent)}")